roboflow classes
1-dump trucks
0-white closed cars
2- excavators

In [1]:
!mkdir dataset

!unzip -q /content/construction.v1i.yolov5pytorch.zip -d /content/dataset

replace /content/dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: y


In [2]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5

%pip install -qr requirements.txt
%pip install -q roboflow

import torch
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

Cloning into 'yolov5'...
remote: Enumerating objects: 17851, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 17851 (delta 31), reused 8 (delta 8), pack-reused 17799 (from 3)
Receiving objects: 100% (17851/17851), 16.98 MiB | 30.39 MiB/s, done.
Resolving deltas: 100% (12161/12161), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 114.6 MB/s eta 0:00:00
Setup complete. Using torch 2.10.0+cu128 (Tesla T4)


In [3]:
import yaml

# Define the path to your yaml file
yaml_path = '/content/dataset/data.yaml'

# Define the new content with absolute paths
data = {
    'train': '/content/dataset/train/images',
    'val': '/content/dataset/valid/images',
    'test': '/content/dataset/test/images',
    'nc': 3,
    'names': ['0', '1', '2']
}

# Write the fixed paths back to the file
with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print("✅ data.yaml updated with absolute paths!")
!cat /content/dataset/data.yaml

✅ data.yaml updated with absolute paths!
names:
- '0'
- '1'
- '2'
nc: 3
test: /content/dataset/test/images
train: /content/dataset/train/images
val: /content/dataset/valid/images


In [4]:
!pip install -U albumentations
!pip install -r https://raw.githubusercontent.com/ultralytics/yolov5/master/requirements.txt

In [5]:
import os
import time
import pandas as pd

start_cmd = "python train.py --img 640 --batch 16 --epochs 60 --data /content/dataset/data.yaml --weights yolov5s.pt --cache"
resume_cmd = start_cmd + " --resume"

results_path = "runs/train/exp/results.csv"

first_run = True

while True:
    current_cmd = start_cmd if first_run else resume_cmd
    print(f"\n--- [EXECUTING]: {current_cmd} ---")

    exit_code = os.system(current_cmd)

    if os.path.exists(results_path):
        try:
            df = pd.read_csv(results_path)
            df.columns = [c.strip() for c in df.columns]
            latest = df.iloc[-1]

            print("\n" + "="*30)
            print(f"LATEST EPOCH STATS:")
            print(f"Precision: {latest['metrics/precision']:.4f}")
            print(f"Recall:    {latest['metrics/recall']:.4f}")
            print(f"mAP@50:    {latest['metrics/mAP_0.5']:.4f}")
            # F1 is usually calculated as 2 * (P*R)/(P+R)
            p = latest['metrics/precision']
            r = latest['metrics/recall']
            f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
            print(f"F1-Score:  {f1:.4f}")
            print("="*30 + "\n")
        except Exception as e:
            print(f"Could not read results yet: {e}")

    if exit_code == 0:
        print("Done! Training reached the final epoch.")
        break
    else:
        print("Session interrupted. Retrying in 10s...")
        first_run = False
        time.sleep(10)


--- [EXECUTING]: python train.py --img 640 --batch 16 --epochs 60 --data /content/dataset/data.yaml --weights yolov5s.pt --cache ---

LATEST EPOCH STATS:
Precision: 0.9254
Recall:    0.9090
mAP@50:    0.9477
F1-Score:  0.9171

Done! Training reached the final epoch.


In [6]:
from google.colab import files
import glob
import os

# 1. Find the newest 'best.pt' weights file in your runs folder
all_weights = glob.glob("runs/train/exp*/weights/best.pt")

if all_weights:
    # Get the most recently modified weights file
    latest_weights = max(all_weights, key=os.path.getmtime)
    print(f"--- [FINISHED] ---")
    print(f"Found your best weights at: {latest_weights}")
    print("Starting download to your device...")

    # Trigger the browser download
    files.download(latest_weights)

    # Optional: Also download the 'last.pt' in case you want to resume training later
    # last_weights = latest_weights.replace("best.pt", "last.pt")
    # files.download(last_weights)
else:
    print("Error: Could not find 'best.pt'. Did the training start correctly?")

--- [FINISHED] ---
Found your best weights at: runs/train/exp/weights/best.pt
Starting download to your device...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>